# Human-in-the-Loop

The Human-in-the-Loop (HITL) pattern strategically integrates human judgment into AI workflows. Agents handle routine cases autonomously, while complex or high-risk situations are escalated to human operators. The agent produces a structured output with an `escalate` flag — the outer orchestration layer decides how to route it.

## Implementation with Flyte v2

This notebook reimplements the ADK technical support agent from Chapter 13 using **Flyte v2 primitives**.

#### ADK vs Flyte v2 — Key Differences

| Aspect | ADK | Flyte v2 |
|--------|-----|----------|
| **Escalation trigger** | `escalate_to_human` tool (side-effecting mock) | `SupportResult(escalate=True)` — structured, side-effect-free |
| **Personalization** | `CallbackContext` injects into LLM request before dispatch | Customer profile passed as typed task input |
| **Ticket creation** | `create_ticket` tool (in-process side effect) | Returns `TicketId` in result — caller decides what to store |
| **State** | Mutable `state` dict shared across agents | Immutable `SupportResult` dataclass |
| **Observability** | `print()` + ADK console | Structured fields (`escalated`, `ticket_id`, `tier`) in Flyte UI |
| **Human approval loop** | Not shown | Outer `while result.escalate` loop in orchestrator |
| **Secrets** | `os.environ` at import time | `flyte.Secret` injected at task runtime |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
import uuid
from dataclasses import dataclass, field
from datetime import timedelta
from typing import Optional

import anthropic
import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="hitl-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0")
)

hitl_env = flyte.TaskEnvironment(
    name="hitl_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define data models

The ADK example passes customer context through a mutable `state` dict injected by a `CallbackContext`. In Flyte v2, `CustomerContext` is a typed input — the agent receives it directly, and `SupportResult` captures the decision as structured output.

The key insight: **escalation is a return value, not a side effect.** The caller (orchestrator) inspects `SupportResult.escalate` and routes accordingly — a human-review queue, a Slack alert, or a follow-up Flyte task.

In [ ]:
@dataclass
class CustomerContext:
    """Replaces ADK's state['customer_info'] injected via CallbackContext."""
    name: str
    tier: str = "standard"           # "standard" | "premium" | "enterprise"
    recent_purchases: list[str] = field(default_factory=list)
    support_history: list[str] = field(default_factory=list)


@dataclass
class SupportResult:
    """Output of the support agent — escalation decision is a typed field, not a side effect."""
    issue: str
    customer_name: str
    troubleshooting_steps: str
    ticket_id: Optional[str]
    escalate: bool
    escalation_reason: str
    agent_response: str

### 5. Define the Anthropic tool schemas

The ADK agent had tools: `troubleshoot_issue`, `create_ticket`, `escalate_to_human`. In Flyte v2, these are Anthropic tool schemas — the LLM decides which to call, and the task executes them. The crucial difference: `escalate_to_human` returns a **signal** rather than performing an action. The outer orchestration loop handles the actual escalation.

In [ ]:
SUPPORT_TOOLS = [
    {
        "name": "troubleshoot_issue",
        "description": "Diagnose the technical issue and return step-by-step troubleshooting instructions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "issue": {"type": "string", "description": "Description of the technical issue."},
                "device_type": {"type": "string", "description": "Type of device (e.g. 'laptop', 'phone')."},
            },
            "required": ["issue"],
        },
    },
    {
        "name": "create_ticket",
        "description": "Create a support ticket and return the ticket ID.",
        "input_schema": {
            "type": "object",
            "properties": {
                "issue_type": {"type": "string", "description": "Category of the issue."},
                "details": {"type": "string", "description": "Full issue description for the ticket."},
            },
            "required": ["issue_type", "details"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": "Signal that this issue exceeds automated troubleshooting and requires a human specialist.",
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string", "description": "Why human intervention is needed."},
                "issue_type": {"type": "string", "description": "Category for routing to the right specialist."},
            },
            "required": ["reason"],
        },
    },
]


def _execute_tool(name: str, inputs: dict) -> tuple[str, dict]:
    """
    Execute a support tool and return (result_text, state_updates).
    Side-effect-free: escalation and tickets are returned in state, not performed here.
    """
    state: dict = {}

    if name == "troubleshoot_issue":
        issue = inputs["issue"]
        device = inputs.get("device_type", "device")
        steps = (
            f"Troubleshooting steps for '{issue}' on {device}:\n"
            "1. Restart the device.\n"
            "2. Check for software updates.\n"
            "3. Clear cache and temporary files.\n"
            "4. Run the built-in diagnostic tool."
        )
        state["troubleshooting_steps"] = steps
        return steps, state

    elif name == "create_ticket":
        ticket_id = f"TKT-{uuid.uuid4().hex[:6].upper()}"
        state["ticket_id"] = ticket_id
        return f"Ticket created: {ticket_id}", state

    elif name == "escalate_to_human":
        state["escalate"] = True
        state["escalation_reason"] = inputs.get("reason", "Requires specialist.")
        return f"Escalation flagged: {inputs.get('reason', '')}", state

    return f"Unknown tool: {name}", state

### 6. Define the HITL support agent task

The ADK agent used `personalization_callback` to inject customer context before each LLM call and had `escalate_to_human` as a side-effecting tool. In Flyte v2:
- Customer context is injected into the **system prompt** directly (no callback needed)
- `escalate_to_human` sets a flag in `state` — the task returns `SupportResult(escalate=True)`
- The **outer orchestration loop** (shown in section 7) checks `escalate` and routes accordingly

In [ ]:
def _build_system_prompt(customer: CustomerContext) -> str:
    """Inject customer context into system prompt — replaces ADK's personalization_callback."""
    purchases = ", ".join(customer.recent_purchases) if customer.recent_purchases else "none"
    history_note = ""
    if customer.support_history:
        history_note = f"\nPast support issues: {'; '.join(customer.support_history[-3:])}"

    return f"""\
You are a technical support specialist for an electronics company.

CUSTOMER CONTEXT:
Name: {customer.name}
Tier: {customer.tier}
Recent purchases: {purchases}{history_note}

For technical issues:
1. Use troubleshoot_issue to diagnose the problem.
2. If unresolved after troubleshooting, use create_ticket to log it.
3. For complex issues beyond basic troubleshooting, use escalate_to_human.

Escalate if: hardware failure suspected, data loss risk, safety concern, or 3+ failed troubleshooting attempts.
Premium/enterprise customers get priority routing."""


@hitl_env.task(
    retries=1,
    timeout=timedelta(minutes=3),
    cache=flyte.Cache(behavior="disable"),
)
async def support_agent(
    issue: str,
    customer: CustomerContext,
    max_steps: int = 8,
) -> SupportResult:
    """
    Technical support agent with HITL escalation.

    Replaces ADK's:
      Agent(name='technical_support_specialist',
            tools=[troubleshoot_issue, create_ticket, escalate_to_human])
      + personalization_callback injecting state['customer_info']

    Escalation is a return value (escalate=True), not a side effect.
    The caller decides how to route escalated cases.
    """
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    system = _build_system_prompt(customer)
    messages = [{"role": "user", "content": issue}]

    # Accumulated state across tool calls
    state: dict = {
        "troubleshooting_steps": "",
        "ticket_id": None,
        "escalate": False,
        "escalation_reason": "",
    }
    final_response = ""

    for _ in range(max_steps):
        response = await client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            system=system,
            tools=SUPPORT_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    final_response = block.text
            break

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result_text, updates = _execute_tool(block.name, block.input)
                    state.update(updates)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_text,
                    })
            messages.append({"role": "user", "content": tool_results})

            # Short-circuit if escalation was decided
            if state["escalate"]:
                break

    return SupportResult(
        issue=issue,
        customer_name=customer.name,
        troubleshooting_steps=state["troubleshooting_steps"],
        ticket_id=state["ticket_id"],
        escalate=state["escalate"],
        escalation_reason=state["escalation_reason"],
        agent_response=final_response,
    )

### 7. Run locally — showing the HITL orchestration loop

The orchestrator checks `result.escalate` and routes accordingly — in production this would send to a Slack channel, PagerDuty, or a human-review queue.

In [ ]:
SCENARIOS = [
    (
        "My laptop screen flickers and sometimes goes black.",
        CustomerContext(
            name="Alice", tier="premium",
            recent_purchases=["ThinkPad X1 Carbon"],
            support_history=[],
        ),
    ),
    (
        "My phone is completely unresponsive after dropping it in water. I can smell burning.",
        CustomerContext(
            name="Bob", tier="standard",
            recent_purchases=["iPhone 15"],
            support_history=["Battery drain", "Screen crack"],
        ),
    ),
    (
        "My Bluetooth headphones won't connect to my laptop.",
        CustomerContext(
            name="Carol", tier="enterprise",
            recent_purchases=["Sony WH-1000XM5"],
            support_history=[],
        ),
    ),
]

for issue, customer in SCENARIOS:
    run = flyte.run(support_agent, issue=issue, customer=customer)
    run.wait()
    result: SupportResult = run.outputs()[0]

    print(f"Customer: {result.customer_name} | Issue: {issue[:60]}")
    print(f"  Ticket: {result.ticket_id or 'none'}")

    if result.escalate:
        # ── HITL routing happens here ─────────────────────────────────────────
        # In production: send to Slack/PagerDuty/human queue
        print(f"  [ESCALATED] → Reason: {result.escalation_reason}")
        print(f"  Action: Routing to human specialist queue...")
    else:
        print(f"  [RESOLVED] Agent response: {result.agent_response[:120]}")
    print()

### Running remotely

The `SupportResult` dataclass makes `escalate`, `escalation_reason`, and `ticket_id` visible as structured output in the Flyte UI — no log parsing needed. In a production pipeline, the orchestrating workflow would check `escalate` and fork to a human-review task (e.g., send a Slack message and wait for approval).

In [ ]:
run = flyte.run(
    support_agent,
    issue="My smart thermostat shows error code E5 and won't heat the house.",
    customer=CustomerContext(name="Dave", tier="standard", recent_purchases=["Nest Thermostat"]),
)
run.wait()
result = run.outputs()[0]
print(f"Escalated: {result.escalate}")
print(f"Ticket: {result.ticket_id}")
print(result.agent_response)